In [ ]:
# %%
# ╔══════════════════════════════════════════════════════════╗
# ║   CELDA 1 — Carga y Optimización de Multi-Bases de Datos ║
# ╚══════════════════════════════════════════════════════════╝
import pandas as pd
import numpy as np

# === 1. CONFIGURACIÓN DE RUTAS Y CONSTANTES ===
PATH_INDICES = 'csv_dashboard/BaseINDICES-2020-2025.csv'
PATH_ING_CHILE = 'csv_dashboard/todas_las_ingenierias_chile.csv'

COL_INST_INDICES = 'Nombre Institución'
COL_REG_INDICES = 'Nombre Region'
COL_CARRERA_INDICES = 'Carrera Genérica'

# === 2. CARGA Y LIMPIEZA: BASE HISTÓRICA INDICES (2020-2025) ===
try:
    df_indices = pd.read_csv(PATH_INDICES, sep=';', encoding='utf-8')
except Exception:
    df_indices = pd.read_csv(PATH_INDICES, sep=',', encoding='utf-8')

cols_num_indices = ['Matrícula primer año hombres', 'Matrícula primer año mujeres', 'Vacantes', 'Matrícula Primer Año', 'Valor de arancel', 'Matrícula Total']
for col in cols_num_indices:
    if col in df_indices.columns:
        if df_indices[col].dtype == 'object':
            df_indices[col] = df_indices[col].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
        df_indices[col] = pd.to_numeric(df_indices[col], errors='coerce').astype('float32')

df_indices['Año'] = pd.to_numeric(df_indices['Año'], errors='coerce').fillna(2024).astype('int16')
df_ing_indices = df_indices[df_indices[COL_CARRERA_INDICES].str.contains('Ingeniería', case=False, na=False)].copy()


# === 3. CARGA Y LIMPIEZA ROBUSTA: BASE NACIONAL ===
df_nacional = None
for encoding_test in ['utf-16', 'latin-1', 'utf-8', 'cp1252']:
    try:
        df_nacional = pd.read_csv(PATH_ING_CHILE, sep=';', encoding=encoding_test)
        break
    except Exception:
        try:
            df_nacional = pd.read_csv(PATH_ING_CHILE, sep=',', encoding=encoding_test)
            break
        except Exception:
            continue

if df_nacional is None:
    raise ValueError("❌ No se pudo decodificar el archivo 'todas_las_ingenierias_chile.csv'. Revisa el formato.")

# Limpieza preventiva de los nombres de columnas
df_nacional.columns = df_nacional.columns.str.strip()

cols_num_nacional = [
    '% Titulados continuidad de estudios', 'Retención de 1er año', 
    'Duración Real (semestres)', 'Empleabilidad al 1er año', 
    'Empleabilidad al 2º Año', 'Ingreso promedio al 4° año'
]

for col in cols_num_nacional:
    if col in df_nacional.columns:
        # 1. Convertimos a string y barremos espacios vacíos
        df_nacional[col] = df_nacional[col].astype(str).str.strip()
        
        # 2. Si viene un rango de sueldos (ej: "De 1.000.000 a 1.100.000"), nos quedamos con el primer tramo
        if col == 'Ingreso promedio al 4° año':
            df_nacional[col] = df_nacional[col].str.split('a').str[0]
            
        # 3. Quitamos los puntos de miles típicos del formato de Excel chileno
        df_nacional[col] = df_nacional[col].str.replace('.', '', regex=False)
        
        # 4. EXPRESIÓN REGULAR: Extrae estrictamente solo los dígitos (remueve $, espacios, letras o guiones)
        df_nacional[col] = df_nacional[col].str.extract(r'(\d+\.?\d*)')[0]
        
        # 5. Pasamos a número flotante real
        df_nacional[col] = pd.to_numeric(df_nacional[col], errors='coerce').astype('float32')


# === 4. 🔥 SISTEMA DE CACHING INDEXADO PARA GRÁFICOS Y ML ===
print("Pre-calculando tablas caché de alto rendimiento...")

# Caché Índices: Para las tendencias y barras regionales
cache_graficos = df_ing_indices.groupby(['Año', COL_CARRERA_INDICES])[['Matrícula primer año hombres', 'Matrícula primer año mujeres', 'Matrícula Total', 'Valor de arancel']].agg({
    'Matrícula primer año hombres': 'sum', 'Matrícula primer año mujeres': 'sum', 'Matrícula Total': 'sum', 'Valor de arancel': 'median'
}).reset_index()

cache_mapas = df_ing_indices.groupby(['Año', COL_REG_INDICES])[['Matrícula Total', 'Matrícula Primer Año', 'Valor de arancel']].agg({
    'Matrícula Total': 'sum', 'Matrícula Primer Año': 'sum', 'Valor de arancel': 'median'
}).reset_index()


# === 5. 🛠️ CONTROL DE CALIDAD ESTRICTO PARA EL CACHÉ DE ML ===
# 1. Filtramos para trabajar únicamente con universidades/carreras que tengan sueldos reales mayores a 0
df_valid_target = df_nacional[df_nacional['Ingreso promedio al 4° año'] > 0].copy()
df_valid_target = df_valid_target.dropna(subset=['Ingreso promedio al 4° año'])

# 2. Guardamos este subset limpio como la base definitiva para los modelos predictivos
cache_kpi_real = df_valid_target.copy()

# 3. Imputamos con la mediana únicamente las características predictoras secundarias (si les faltara algún dato)
for col in cols_num_nacional:
    if col != 'Ingreso promedio al 4° año' and col in cache_kpi_real.columns:
        mediana_col = cache_kpi_real[col].median()
        if not np.isnan(mediana_col):
            cache_kpi_real[col] = cache_kpi_real[col].fillna(mediana_col)

print(f"✅ DATA STORE CONTROL: Bases sincronizadas de forma segura.")
print(f"-> Histórico: {len(df_ing_indices)} registros | Nacional (ML Ready): {len(cache_kpi_real)} registros con datos reales.")

SyntaxError: invalid syntax (4110727520.py, line 52)

In [ ]:
# %%
# %%
# ╔══════════════════════════════════════════════════════════╗
# ║  CELDA 2 — Diseño de la Interfaz Visual y Contenedores    ║
# ╚══════════════════════════════════════════════════════════╝
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go
import plotly.express as px

# 1. Inyección de estilos CSS para forzar bordes redondos en los botones
estilos_css = widgets.HTML("""
<style>
    .tabs-redondeadas .btn {
        border-radius: 16px !important; 
        margin-right: 6px !important;   
        border: 1px solid #bce8f1 !important;
    }
</style>
""")

# 2. Barra de Navegación Superior
tabs_navegacion = widgets.ToggleButtons(
    options=['KPIs', 'GRÁFICOS', 'MAPAS', 'ML (RANDOM FOREST PESOS)'],
    value='KPIs',
    button_style='info',
    layout=widgets.Layout(width='100%', margin='0px 0px 5px 0px')
)
tabs_navegacion.add_class('tabs-redondeadas') 

linea_separadora = widgets.HTML("<hr style='border: 0; border-top: 3px solid #000000; margin: 5px 0px 15px 0px; width: 100%; opacity: 1;'>")


# =====================================================================
# MAQUETACIÓN INTERNA: PESTAÑA 'KPIs' (DOS SELECTORES IZQ / TARJETAS DER)
# =====================================================================
selector_kpi_institucion = widgets.Dropdown(
    options=['---', 'Universidad de Chile', 'Pontificia Universidad Catolica', 'Universidad de Concepcion', 'Universidad Austral de Chile'],
    value='---',
    description='Institucion:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='95%', margin='10px 0px')
)

selector_kpi_carrera = widgets.Dropdown(
    options=['---', 'Ingenieria Civil Informatica', 'Ingenieria Civil Industrial', 'Ingenieria Civil en Obras Civiles', 'Ingenieria Comercial'],
    value='---',
    description='Carrera:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='95%', margin='5px 0px 15px 0px')
)

panel_izquierdo_kpis = widgets.VBox([
    widgets.HTML("<h4>Filtros KPI</h4><hr style='margin:5px 0; border-top:1px solid #ccc;'>"),
    selector_kpi_institucion,
    selector_kpi_carrera,
    widgets.HTML("<div style='color:#777; font-size:0.85em; padding-top:15px; font-family:sans-serif; line-height:1.3;'>"
                 "<i>Selecciona una institucion y una carrera especifica para cargar los indicadores clave de rendimiento asociados.</i></div>")
], layout=widgets.Layout(width='280px', padding='15px', border='1px solid #ccc', margin='0px 15px 0px 0px', bg_color='#fbfbfb'))

area_kpi_cards = widgets.Output(
    layout=widgets.Layout(width='620px', height='410px', border='1px solid #eee', bg_color='#fafafa', padding='10px')
)

layout_tab_kpis = widgets.HBox([panel_izquierdo_kpis, area_kpi_cards], layout=widgets.Layout(width='100%', height='100%', padding='5px'))


# =====================================================================
# MAQUETACIÓN INTERNA: PESTAÑA 'GRÁFICOS'
# =====================================================================
diccionario_graficos = {
    '---': '',
    'Brecha de Genero en Matematicas (M1 vs M2)': 'csv/boxplot_genero_m1m2.png',
    'Distribucion por Rama Educacional (HC vs TP)': 'csv/violin_rama_educacional.png',
    'Impacto del Programa PACE': 'csv/bar_pace_impact.png',
    'Relacion Competencia Lectora vs Matematicas': 'csv/jointplot_lect_mate.png',
    'Evolucion del Puntaje por Dependencia': 'csv/lineplot_gap_evolution.png'
}

selector_graficos = widgets.Dropdown(
    options=list(diccionario_graficos.keys()),
    value='---',
    description='Grafico:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='95%', margin='10px 0px')
)

selector_dimensiones = widgets.Dropdown(
    options=['---', 'Genero', 'Dependencia', 'Rama Educacional', 'PACE'],
    value='---',
    description='Dimensiones:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='95%', margin='10px 0px')
)

panel_izquierdo_graficos = widgets.VBox([
    widgets.HTML("<h4>Reportes Estadisticos</h4><hr style='margin:5px 0; border-top:1px solid #ccc;'>"),
    selector_graficos,
    selector_dimensiones,
    widgets.HTML("<div style='color:#777; font-size:0.85em; padding-top:15px; font-family:sans-serif; line-height:1.3;'>"
                 "<i>Selecciona un reporte estadistico y su dimension para cargar el analisis exploratorio de datos.</i></div>")
], layout=widgets.Layout(width='280px', padding='15px', border='1px solid #ccc', margin='0px 15px 0px 0px', bg_color='#fbfbfb'))

area_imagen_grafico = widgets.Output(
    layout=widgets.Layout(width='620px', height='410px', border='1px solid #eee', bg_color='#fafafa', padding='10px')
)

layout_tab_graficos = widgets.HBox([panel_izquierdo_graficos, area_imagen_grafico], layout=widgets.Layout(width='100%', height='100%', padding='5px'))


# =====================================================================
# MAQUETACIÓN INTERNA: PESTAÑA 'MAPAS'
# =====================================================================
diccionario_mapas = {
    '---': '',
    'Mapa de Empleabilidad Regional': 'csv/mapa_empleabilidad_2025.png',
    'Mapa de Retencion de Primer Año': 'csv/mapa_retencion_2025.png',
    'Mapa de Ingresos Promedio': 'csv/mapa_ingresos_2025.png'
}

selector_mapas = widgets.Dropdown(
    options=list(diccionario_mapas.keys()),
    value='---',
    description='Ver mapa:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='95%', margin='10px 0px')
)

panel_izquierdo_mapas = widgets.VBox([
    widgets.HTML("<h4>Variables</h4><hr style='margin:5px 0; border-top:1px solid #ccc;'>"),
    selector_mapas,
    widgets.HTML("<div style='color:#777; font-size:0.85em; padding-top:15px; font-family:sans-serif; line-height:1.3;'>"
                 "<i>Elige una opcion en el menu superior para actualizar el visor del mapa regional.</i></div>")
], layout=widgets.Layout(width='280px', padding='15px', border='1px solid #ccc', margin='0px 15px 0px 0px', bg_color='#fbfbfb'))

area_imagen_mapa = widgets.Output(
    layout=widgets.Layout(width='620px', height='410px', border='1px solid #eee', bg_color='#fafafa', padding='10px')
)

layout_tab_mapas = widgets.HBox([panel_izquierdo_mapas, area_imagen_mapa], layout=widgets.Layout(width='100%', height='100%', padding='5px'))


# =====================================================================
# PESTAÑA 'ML' - NUEVA ESTRUCTURA CON SELECTORES
# =====================================================================
selector_modelo_ml = widgets.Dropdown(
    options=['Regresión Lineal', 'K-Means Clustering', 'PCA Análisis'],
    value='Regresión Lineal',
    description='Modelo ML:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='95%', margin='10px 0px')
)

selector_sub_analisis_ml = widgets.Dropdown(
    options=['---'],
    value='---',
    description='Sub-análisis:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='95%', margin='10px 0px')
)

panel_izquierdo_ml = widgets.VBox([
    widgets.HTML("<h4>Machine Learning</h4><hr style='margin:5px 0; border-top:1px solid #ccc;'>"),
    selector_modelo_ml,
    selector_sub_analisis_ml,
    widgets.HTML("<div style='color:#777; font-size:0.85em; padding-top:15px; font-family:sans-serif; line-height:1.3;'>"
                 "<i>Selecciona un modelo y su análisis correspondiente para explorar patrones predictivos.</i></div>")
], layout=widgets.Layout(width='280px', padding='15px', border='1px solid #ccc', margin='0px 15px 0px 0px', bg_color='#fbfbfb'))

area_imagen_ml = widgets.Output(
    layout=widgets.Layout(width='620px', height='410px', border='1px solid #eee', bg_color='#fafafa', padding='10px')
)

layout_tab_ml = widgets.HBox([panel_izquierdo_ml, area_imagen_ml], layout=widgets.Layout(width='100%', height='100%', padding='5px'))


# Contenedor dinamico central para el intercambio de vistas
contenedor_cuerpo = widgets.Output(layout=widgets.Layout(width='100%', height='430px', overflow='auto'))

print("Celda 2: Arquitectura visual de la interfaz cargada en la memoria.")

Celda 2: Arquitectura visual de la interfaz cargada en la memoria.


In [ ]:
# %%
# ╔══════════════════════════════════════════════════════════╗
# ║  CELDA 3 — Lógica y Renderizado con Datos Reales + ML    ║
# ╚══════════════════════════════════════════════════════════╝
import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt
import numpy as np
import sys
sys.path.insert(0, '.')  

# ===== IMPORTAR MÓDULOS ML =====
from rlm import entrenar_regresion_ingresos
from kmeans import clustering_carreras
from pca import analisis_pca_completo

ANCHO_FIJO = '980px'
ALTO_FIJO = '580px'

# 1. Enrutador general de pestañas superiores
def alternar_pestanas(change):
    with contenedor_cuerpo:
        clear_output(wait=True)
        pestana_activa = change['new'] if change else tabs_navegacion.value
        
        if pestana_activa == 'KPIs':
            display(layout_tab_kpis)
            actualizar_kpi_cards(None)
        elif pestana_activa == 'GRÁFICOS':
            display(layout_tab_graficos)
            actualizar_imagen_grafico(None)
        elif pestana_activa == 'MAPAS':
            display(layout_tab_mapas)
            actualizar_imagen_mapa(None)
        elif pestana_activa == 'ML (RANDOM FOREST PESOS)':
            display(layout_tab_ml)
            actualizar_opciones_sub_analisis(None) # Dispara la cadena de actualización de ML

tabs_navegacion.observe(alternar_pestanas, names='value')


# ===== MODELOS ML: ACTUALIZAR OPCIONES DE SUB-ANÁLISIS =====
def actualizar_opciones_sub_analisis(change):
    modelo = selector_modelo_ml.value
    
    if modelo == 'Regresión Lineal':
        opciones = ['Predicción vs Actual', 'Análisis Residuales', 'Importancia Features']
    elif modelo == 'K-Means Clustering':
        opciones = ['Elbow Method', 'Clusters 2D (PCA)', 'Clusters 3D (PCA)']
    elif modelo == 'PCA Análisis':
        opciones = ['Scree Plot', 'Biplot 2D', 'Loading Heatmap', 'PCA 3D']
    
    # Bloqueamos temporalmente el observe para evitar doble ejecución al cambiar opciones
    selector_sub_analisis_ml.unobserve(actualizar_sub_analisis_ml, names='value')
    selector_sub_analisis_ml.options = opciones
    selector_sub_analisis_ml.value = opciones[0] if opciones else '---'
    selector_sub_analisis_ml.observe(actualizar_sub_analisis_ml, names='value')
    
    # Forzamos el renderizado con el nuevo set de datos
    actualizar_sub_analisis_ml(None)

selector_modelo_ml.observe(actualizar_opciones_sub_analisis, names='value')


# ===== MOTOR DE GRÁFICOS ML =====
def actualizar_sub_analisis_ml(change):
    with area_imagen_ml:
        clear_output(wait=True)
        
        modelo = selector_modelo_ml.value
        sub_analisis = selector_sub_analisis_ml.value
        
        if sub_analisis == '---' or not sub_analisis:
            return
            
        # ===== REGRESIÓN LINEAL =====
        if modelo == 'Regresión Lineal':
            resultado_regresion = entrenar_regresion_ingresos(cache_kpi_real)
            if resultado_regresion['error']:
                display(widgets.HTML(f"<div style='padding:20px; color:#d9534f;'><b>❌ Error:</b> {resultado_regresion['error']}</div>"))
            else:
                metricas = resultado_regresion['metricas']
                html_metricas = f"""
                <div style='background-color: #f0f8ff; border-left: 4px solid #5cb85c; padding: 10px; margin-bottom: 10px; font-size: 0.9em; font-family: monospace;'>
                    <b>Métricas:</b> R²={metricas.get('R² Test', 0):.4f} | RMSE=${metricas.get('RMSE Test', 0):,.0f} | MAE=${metricas.get('MAE Test', 0):,.0f}
                </div>
                """
                display(widgets.HTML(html_metricas))
                if sub_analisis == 'Predicción vs Actual': resultado_regresion['figura_prediccion'].show()
                elif sub_analisis == 'Análisis Residuales': resultado_regresion['figura_residuales'].show()
                elif sub_analisis == 'Importancia Features': resultado_regresion['figura_importancia'].show()
        
        # ===== K-MEANS CLUSTERING =====
        elif modelo == 'K-Means Clustering':
            resultado_kmeans = clustering_carreras(cache_kpi_real)
            if resultado_kmeans['error']:
                display(widgets.HTML(f"<div style='padding:20px; color:#d9534f;'><b>❌ Error:</b> {resultado_kmeans['error']}</div>"))
            else:
                metricas = resultado_kmeans['metricas']
                html_metricas = f"""
                <div style='background-color: #fffacd; border-left: 4px solid #f0ad4e; padding: 10px; margin-bottom: 10px; font-size: 0.9em; font-family: monospace;'>
                    <b>Clustering:</b> K={metricas.get('K_optimo', 0)} | Varianza 2D={metricas.get('varianza_explicada_2d', 0):.1f}%
                </div>
                """
                display(widgets.HTML(html_metricas))
                if sub_analisis == 'Elbow Method': resultado_kmeans['figura_elbow'].show()
                elif sub_analisis == 'Clusters 2D (PCA)': resultado_kmeans['figura_clusters_2d'].show()
                elif sub_analisis == 'Clusters 3D (PCA)': resultado_kmeans['figura_clusters_3d'].show()
        
        # ===== PCA ANÁLISIS =====
        elif modelo == 'PCA Análisis':
            resultado_pca = analisis_pca_completo(cache_kpi_real)
            if resultado_pca['error']:
                display(widgets.HTML(f"<div style='padding:20px; color:#d9534f;'><b>❌ Error:</b> {resultado_pca['error']}</div>"))
            else:
                metricas = resultado_pca['metricas']
                html_metricas = f"""
                <div style='background-color: #f0f8f8; border-left: 4px solid #5bc0de; padding: 10px; margin-bottom: 10px; font-size: 0.9em; font-family: monospace;'>
                    <b>PCA:</b> Componentes={metricas.get('n_componentes_80_varianza', 0)} | Varianza 3D={metricas.get('varianza_explicada_3d', 0):.1f}%
                </div>
                """
                display(widgets.HTML(html_metricas))
                if sub_analisis == 'Scree Plot': resultado_pca['figura_scree'].show()
                elif sub_analisis == 'Biplot 2D': resultado_pca['figura_biplot'].show()
                elif sub_analisis == 'Loading Heatmap': resultado_pca['figura_loadings_heatmap'].show()
                elif sub_analisis == 'PCA 3D': resultado_pca['figura_pca_3d'].show()

# Escuchar correctamente los cambios en el menú de sub-análisis
selector_sub_analisis_ml.observe(actualizar_sub_analisis_ml, names='value')


# 2. MOTOR DE KPIs REALES
def actualizar_kpi_cards(change):
    with area_kpi_cards:
        clear_output(wait=True)
        institucion = selector_kpi_institucion.value
        carrera = selector_kpi_carrera.value
        
        if institucion == '---' or carrera == '---':
            display(widgets.HTML("<div style='display:flex; justify-content:center; align-items:center; height:100%; width:100%; color:#999; font-style:italic; font-family:sans-serif; text-align:center; padding-top:140px;'>"
                                 "<h4>[ Selecciona una institucion y una carrera para desplegar las metricas KPI ]</h4></div>"))
        else:
            m_inst = institucion.lower().replace('pontificia ', '').split(' de ')[0]
            dic_carreras = {
                'Ingenieria Civil Informatica': 'informática|computación|software',
                'Ingenieria Civil Industrial': 'industrial',
                'Ingenieria Civil en Obras Civiles': 'obras civiles|civil',
                'Ingenieria Comercial': 'comercial'
            }
            m_carr = dic_carreras.get(carrera, 'invalid')
            
            df_res = cache_kpi_real[
                (cache_kpi_real['Institución'].str.lower().str.contains(m_inst, na=False)) &
                (cache_kpi_real['Carrera'].str.lower().str.contains(m_carr, na=False))
            ]
            
            if not df_res.empty:
                fila = df_res.iloc[0]
                v_ret = f"{fila['Retención de 1er año']:.1f}%" if not np.isnan(fila['Retención de 1er año']) else "--%"
                v_emp = f"{fila['Empleabilidad al 2º Año']:.1f}%" if not np.isnan(fila['Empleabilidad al 2º Año']) else "--%"
                v_dur = f"{fila['Duración Real (semestres)']:.1f} sem" if not np.isnan(fila['Duración Real (semestres)']) else "-- sem"
                ingreso_num = fila['Ingreso promedio al 4° año']
                v_ingreso = f"${int(ingreso_num):,}".replace(',', '.') if not np.isnan(ingreso_num) else "No disponible"
            else:
                v_ret, v_emp, v_dur, v_ingreso = "--%", "--%", "-- sem", "--"
                
            html_content = f"""
            <div style='font-family: sans-serif; padding: 5px; height:100%;'>
                <h4 style='color: #2c3e50; margin-top: 0; margin-bottom: 5px;'>Datos Oficiales Mifuturo: {institucion}</h4>
                <h5 style='color: #555; margin-top: 0; margin-bottom: 15px; font-weight: normal;'>Programa: <b>{carrera}</b></h5>
                <div style='display: flex; justify-content: space-between; gap: 10px; margin-bottom: 15px;'>
                    <div style='flex: 1; background-color: #f8f9fa; border-left: 5px solid #5bc0de; padding: 10px; border-radius: 4px; text-align: center;'>
                        <div style='font-size: 0.78em; color: #777; font-weight: bold; text-transform: uppercase;'>Retención</div>
                        <div style='font-size: 1.4em; font-weight: bold; color: #333; margin-top:4px;'>{v_ret}</div>
                    </div>
                    <div style='flex: 1; background-color: #f8f9fa; border-left: 5px solid #5cb85c; padding: 10px; border-radius: 4px; text-align: center;'>
                        <div style='font-size: 0.78em; color: #777; font-weight: bold; text-transform: uppercase;'>Empleabilidad</div>
                        <div style='font-size: 1.4em; font-weight: bold; color: #333; margin-top:4px;'>{v_emp}</div>
                    </div>
                    <div style='flex: 1; background-color: #f8f9fa; border-left: 5px solid #f0ad4e; padding: 10px; border-radius: 4px; text-align: center;'>
                        <div style='font-size: 0.78em; color: #777; font-weight: bold; text-transform: uppercase;'>Duración Real</div>
                        <div style='font-size: 1.4em; font-weight: bold; color: #333; margin-top:4px;'>{v_dur}</div>
                    </div>
                </div>
                <div style='background-color: #eef9f0; border: 1px solid #c3e6cb; border-left: 6px solid #28a745; padding: 12px; border-radius: 4px; text-align: center;'>
                    <div style='font-size: 0.85em; color: #155724; font-weight: bold; text-transform: uppercase;'>💰 Sueldo Promedio Estimado al 4° Año</div>
                    <div style='font-size: 1.8em; font-weight: bold; color: #1e7e34; margin-top: 5px;'>{v_ingreso}</div>
                </div>
            </div>
            """
            display(widgets.HTML(html_content))

selector_kpi_institucion.observe(actualizar_kpi_cards, names='value')
selector_kpi_carrera.observe(actualizar_kpi_cards, names='value')


# 3. MOTOR DE MAPAS
def actualizar_imagen_mapa(change):
    with area_imagen_mapa:
        clear_output(wait=True)
        opcion_seleccionada = selector_mapas.value
        
        if opcion_seleccionada == '---':
            display(widgets.HTML("<div style='display:flex; justify-content:center; align-items:center; height:100%; width:100%; color:#999; font-style:italic; font-family:sans-serif; text-align:center; padding-top:140px;'>"
                                 "<h4>[ Elige una opción en el menú izquierdo para renderizar el reporte regional ]</h4></div>"))
        else:
            df_geo = cache_mapas[cache_mapas['Año'] == cache_mapas['Año'].max()]
            if opcion_seleccionada == 'Mapa de Empleabilidad Regional':
                fig = px.bar(df_geo.sort_values('Matrícula Total', ascending=True), x='Matrícula Total', y=COL_REG_INDICES, orientation='h', color='Matrícula Total', color_continuous_scale='Viridis')
            elif opcion_seleccionada == 'Mapa de Retencion de Primer Año':
                fig = px.bar(df_geo.sort_values('Matrícula Primer Año', ascending=True), x='Matrícula Primer Año', y=COL_REG_INDICES, orientation='h', color='Matrícula Primer Año', color_continuous_scale='Plasma')
            elif opcion_seleccionada == 'Mapa de Ingresos Promedio':
                fig = px.bar(df_geo.sort_values('Valor de arancel', ascending=True), x='Valor de arancel', y=COL_REG_INDICES, orientation='h', color='Valor de arancel', color_continuous_scale='Coolwarm')
                fig.update_layout(xaxis_tickformat="$")
                
            fig.update_layout(height=380, width=590, margin=dict(l=10, r=10, t=30, b=10), paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)')
            fig.show()

selector_mapas.observe(actualizar_imagen_mapa, names='value')


# 4. MOTOR DE GRÁFICOS INTERACTIVOS
def actualizar_imagen_grafico(change):
    with area_imagen_grafico:
        clear_output(wait=True)
        opcion_seleccionada = selector_graficos.value
        
        if opcion_seleccionada == '---':
            display(widgets.HTML("<div style='display:flex; justify-content:center; align-items:center; height:100%; width:100%; color:#999; font-style:italic; font-family:sans-serif; text-align:center; padding-top:140px;'>"
                                 "<h4>[ Selecciona un reporte del panel izquierdo para desplegar el grafico ]</h4></div>"))
        else:
            if opcion_seleccionada == 'Brecha de Genero en Matematicas (M1 vs M2)':
                df_g = cache_graficos.groupby('Año')[['Matrícula primer año hombres', 'Matrícula primer año mujeres']].sum().reset_index()
                df_g['Porcentaje Mujeres (%)'] = (df_g['Matrícula primer año mujeres'] / (df_g['Matrícula primer año hombres'] + df_g['Matrícula primer año mujeres'])) * 100
                fig = px.line(df_g, x='Año', y='Porcentaje Mujeres (%)', markers=True)
                fig.update_yaxes(range=[0, 50])
            elif opcion_seleccionada == 'Distribucion por Rama Educacional (HC vs TP)':
                df_g = cache_graficos.groupby('Año')['Matrícula Total'].sum().reset_index()
                fig = px.line(df_g, x='Año', y='Matrícula Total', markers=True)
            elif opcion_seleccionada == 'Evolucion del Puntaje por Dependencia':
                df_g = cache_graficos.groupby('Año')['Valor de arancel'].median().reset_index()
                fig = px.line(df_g, x='Año', y='Valor de arancel', markers=True)
                fig.update_layout(yaxis_tickformat="$")
            else:
                return

            fig.update_layout(height=380, width=590, margin=dict(l=10, r=10, t=30, b=10), paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)')
            fig.show()

selector_graficos.observe(actualizar_imagen_grafico, names='value')


# === 5. LOGIN Y SEGURIDAD COMPARTIDA ===
txt_usuario = widgets.Text(description='Usuario:', placeholder='admin', layout=widgets.Layout(margin='5px 0px', width='280px'))
txt_password = widgets.Password(description='Clave:', placeholder='admin', layout=widgets.Layout(margin='5px 0px', width='280px'))
btn_login = widgets.Button(description='Autenticar', button_style='primary', icon='lock', layout=widgets.Layout(margin='20px 0px 5px 0px', width='280px'))
html_feedback = widgets.HTML(value="")

formulario_interno = widgets.VBox([
    widgets.HTML("<h3 style='text-align: center; font-family: sans-serif; color: #333; margin-top:0;'>SISTEMA DE ACCESO</h3><hr style='width: 100%; border: 0; border-top: 1px solid #ccc;'>"),
    txt_usuario, txt_password, btn_login, html_feedback
], layout=widgets.Layout(width='360px', padding='25px', border='1px solid #ccc', align_items='center', border_radius='4px'))
formulario_interno.add_class('bg-login')

cuadro_login = widgets.VBox([formulario_interno], layout=widgets.Layout(width=ANCHO_FIJO, height=ALTO_FIJO, border='3px solid #333', justify_content='center', align_items='center'))
cuadro_login.add_class('bg-main')

dashboard_final = widgets.VBox([estilos_css, tabs_navegacion, linea_separadora, contenedor_cuerpo], layout=widgets.Layout(width=ANCHO_FIJO, height=ALTO_FIJO, border='3px solid #333', padding='20px'))
dashboard_final.add_class('bg-main')

def validar_credenciales(b):
    if txt_usuario.value == 'admin' and txt_password.value == 'admin':
        with lienzo_maestro:
            clear_output()
            display(dashboard_final)
            alternar_pestanas(None)
    else:
        txt_password.value = ""
        html_feedback.value = "<div style='color: #d9534f; font-weight: bold; text-align: center; margin-top: 12px; font-family: sans-serif;'>Error: Credenciales Incorrectas</div>"

btn_login.on_click(validar_credenciales)
lienzo_maestro = widgets.Output()

print("Celda 3: Motores reactivos cruzados con la base nacional + ML ejecutados con éxito.")

Celda 3: Motores reactivos cruzados con la base nacional + ML ejecutados con éxito.


In [ ]:
# %%
# %%
# ╔══════════════════════════════════════════════════════════╗
# ║  CELDA 4 — Inicialización y Lanzamiento en Pantalla       ║
# ╚══════════════════════════════════════════════════════════╝

# 1. Desplegamos el nodo de salida raíz (el lienzo maestro) en Jupyter
display(lienzo_maestro)

# 2. Forzamos el renderizado inicial de la caja de login dentro del lienzo
with lienzo_maestro:
    clear_output()
    display(cuadro_login)

Output()

In [ ]:
# Celda de diagnóstico rápido
print("--- Diagnóstico de la columna de Ingresos ---")
print(f"Columnas disponibles en el CSV: {list(df_nacional.columns)}")
print("\nPrimeros 5 valores crudos de la columna de ingresos:")
if 'Ingreso promedio al 4° año' in df_nacional.columns:
    print(df_nacional['Ingreso promedio al 4° año'].head(10))
    print("\nTipos de datos actuales:")
    print(df_nacional['Ingreso promedio al 4° año'].dtypes)
    print(f"\nCantidad de filas en df_nacional: {len(df_nacional)}")
    print(f"Cantidad de filas con ingresos mayores a 0: {len(df_nacional[df_nacional['Ingreso promedio al 4° año'] > 0])}")
else:
    print("⚠️ ¡La columna 'Ingreso promedio al 4° año' no existe con ese nombre exacto!")

--- Diagnóstico de la columna de Ingresos ---
Columnas disponibles en el CSV: ['Institución', 'Acreditación institución', 'Carrera', '% Titulados continuidad de estudios', 'Retención de 1er año', 'Duración Real (semestres)', 'Empleabilidad al 1er año', 'Empleabilidad al 2º Año', 'Ingreso promedio al 4° año']

Primeros 5 valores crudos de la columna de ingresos:
0   NaN
1   NaN
2   NaN
3   NaN
4   NaN
5   NaN
6   NaN
7   NaN
8   NaN
9   NaN
Name: Ingreso promedio al 4° año, dtype: float32

Tipos de datos actuales:
float32

Cantidad de filas en df_nacional: 306
Cantidad de filas con ingresos mayores a 0: 0
